In [5]:
from pathlib import Path
import os
import boto3
from dotenv import load_dotenv, find_dotenv

# === BASE DIR (depuis notebooks/) ===
BASE_DIR = Path.cwd().resolve().parent

# === PATHS ===
RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "data" / "outputs"

COORDINATES_PATH = RAW_DIR / "coordinates" / "coordinates.json"
RAW_OUTPUT_PATH = RAW_DIR / "weather" / "weather_forecasts.json"
CSV_OUTPUT_PATH = RAW_DIR / "weather" / "weather_forecasts.csv"

# === ENV ===
load_dotenv(find_dotenv(), override=True)

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")  # ⚠️ espace corrigé
AWS_REGION = "eu-west-3"
BUCKET_NAME = "01-kayak-jedha"

# === S3 ===
def get_s3_resource():
    session = boto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_REGION,
    )
    return session.resource("s3")


def upload_file(local_path: Path, s3_key: str):
    s3 = get_s3_resource()
    bucket = s3.Bucket(BUCKET_NAME)
    bucket.upload_file(str(local_path), s3_key)
    print(f"Upload {local_path} -> s3://{BUCKET_NAME}/{s3_key}")


def upload_raw_folder(prefix: str = "raw/"):
    for path in RAW_DIR.rglob("*.csv"):  # prend aussi sous-dossiers
        s3_key = f"{prefix}{path.name}"
        upload_file(path, s3_key)


def upload_outputs_folder(prefix: str = "outputs/"):
    for path in OUTPUT_DIR.rglob("*.csv"):
        s3_key = f"{prefix}{path.name}"
        upload_file(path, s3_key)


def upload_all():
    upload_raw_folder()
    upload_outputs_folder()


# === RUN ===
upload_all()

Upload /Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/Bloc_01_Kayak_Project_2026/data/raw/booking/booking_data.csv -> s3://01-kayak-jedha/raw/booking_data.csv
Upload /Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/Bloc_01_Kayak_Project_2026/data/raw/weather/weather_forecasts.csv -> s3://01-kayak-jedha/raw/weather_forecasts.csv
